In [ ]:
from langchain.messages import HumanMessage
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# 1.构建大语言模型
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {
            "type": "disabled"
        }
    }
)

# 2.声明状态
class OverAllState(MessagesState):
    output: str

# 3.声明结点
def llm_node(state: OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages": [res]
    }

def output_node(state: OverAllState) -> OverAllState:
    return {
        "output": state["messages"][-1].content
    }

# 4.构建图
builder = StateGraph(state_schema = OverAllState)
builder.add_node("llm_node", llm_node)
builder.add_node("output_node", output_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "output_node")
builder.add_edge("output_node", END)

# 5.配置检查点存储器
checkpointer = InMemorySaver()
graph =  builder.compile(
    checkpointer = checkpointer
)

# 6.使用的时候必须填写线程ID
config = {
    "configurable": {
        "thread_id": "chapter03-01"
    }
}
# 7.执行图
graph.invoke({"messages":[HumanMessage(content = "你好，我是江来")]}, config = config)

In [ ]:
# 8.再次运行，测试持久化记忆
graph.invoke({"messages":[HumanMessage(content = "你好，我是谁呢？")]}, config = config)

In [ ]:
# 9.给定不同config测试持久化记忆
config1 = {
    "configurable": {
        "thread_id": "666"
    }
}

graph.invoke({"messages":[HumanMessage(content = "你好，我是谁呢？")]}, config = config1)